# Pre-etiquetado de lotes (SAM 3)
Procesa los `lote_*.zip` que deja `python xd.py preetiquetar N` en Drive (`PoC-Labelling/lotes`) y escribe
`<lote>_pred.json`. Despues, en el host: `python xd.py lote-importar`.
1. Entorno de ejecucion: **T4 GPU**.
2. Celda 1 una vez por sesion. Celda 2 cada vez que haya lotes.

Versiones: las mismas de `motor/requirements.txt` (lo verifica `test_notebooks`).

In [ ]:
!pip -q install ultralytics==8.4.174 timm==1.0.30 "clip @ git+https://github.com/ultralytics/CLIP.git@7ffa84b3bfa40c42ecc1c77147a855e69cb2dd40"
from google.colab import drive
drive.mount('/content/drive')
!cp -n /content/drive/MyDrive/PoC-Labelling/sam3.pt /content/sam3.pt
!ls -lh /content/sam3.pt /content/drive/MyDrive/PoC-Labelling/lotes

In [ ]:
import json, sys, zipfile
from pathlib import Path

LOTES = Path('/content/drive/MyDrive/PoC-Labelling/lotes')
CACHE = globals().get('CACHE', {})          # predictores entre ejecuciones (nunca dos SAM 3 a la vez)

for zp in sorted(LOTES.glob('lote_*.zip')):
    salida = zp.with_name(zp.stem + '_pred.json')
    if salida.exists():
        print('ya procesado:', zp.name); continue
    d = Path('/content/lotes') / zp.stem
    with zipfile.ZipFile(zp) as z:
        z.extractall(d)
    if not (d / 'codigo' / 'lote.py').exists():
        print('SIN lote.py (lote anterior a D.6): reexportalo ->', zp.name); continue
    for m in [p.stem for p in (d / 'codigo').glob('*.py')]:     # siempre el codigo que trae el zip
        sys.modules.pop(m, None)
    sys.path.insert(0, str(d / 'codigo'))
    try:
        import lote
        out = lote.procesar(d, CACHE, zp.name)
    finally:
        sys.path.pop(0)
    tmp = salida.with_suffix('.tmp')                               # nunca un _pred.json a medias
    tmp.write_text(json.dumps(out), encoding='utf-8')
    tmp.replace(salida)
    print('->', salida.name)